In [ ]:
print("Version de Spark :", spark.version)
print("Application ID   :", spark.sparkContext.applicationId)

In [ ]:
spark.range(1000).count()

In [ ]:
from pyspark.sql import functions as F

h = spark.sql("DESCRIBE HISTORY silver_weather")
display(h.select(
    "version",
    "timestamp",
    "operation",
    F.col("operationMetrics")["numSourceRows"].alias("lignes_source"),
    F.col("operationMetrics")["numTargetRowsInserted"].alias("inserees"),
    F.col("operationMetrics")["numTargetRowsUpdated"].alias("mises_a_jour"),
    F.col("operationMetrics")["executionTimeMs"].alias("duree_ms"),
).orderBy(F.col("version").desc()).limit(5))

In [ ]:
spark.sql("SELECT COUNT(*) AS lignes FROM silver_weather VERSION AS OF 0").show()
spark.sql("SELECT COUNT(*) AS lignes FROM silver_weather").show()

In [ ]:
spark.table("silver_quality_log").orderBy("run_ts", ascending=False).show(3, False)

In [ ]:
from pyspark.sql import functions as F
(spark.sql("DESCRIBE HISTORY silver_weather")
   .select("version", "timestamp", "operation",
           F.col("operationMetrics")["numTargetRowsInserted"].alias("inserees"),
           F.col("operationMetrics")["numTargetRowsUpdated"].alias("mises_a_jour"))
   .orderBy(F.col("version").desc()).show(5, False))

In [ ]:
spark.sql("DESCRIBE DETAIL silver_weather").select("numFiles", "sizeInBytes").show()

In [ ]:
spark.table("silver_quality_log").orderBy("run_ts", ascending=False) \
     .select("run_ts", "files_read", "rows_written", "missing_cities").show(10, False)

In [ ]:
import re
from collections import Counter
names = [f.name for f in notebookutils.fs.ls("Files/bronze/weather") if f.name.endswith(".json")]
c = Counter(re.search(r"_(\d{10})\d{2}\.json", n).group(1) for n in names)  # aaaammjjhh
for k in sorted(c)[-10:]:
    print(k, c[k])